# 2. How Models Are Made

To build and edit your own computational models, you need to understand how a biological diagram turns into a mathematical model. Take your time with this section!

## From a diagram to a system of ODEs

A computational model is a **system of equations**:
- **"System"** because there are multiple coupled equations — one for each biomolecular species.
- **"Equations"** because each one relates the *rate of change* of a species to the reactions affecting it.

Let's look at our Toy Model diagram again:

```{figure} ../images/toy-model-abc.svg
:alt: Toy Model diagram A to B to C
:width: 100%
```

Focus on protein **A**. Two arrows touch **A**:
- **Reaction 1** points *into* **A** (expression at constant rate $k_{1,\text{Aexp}}$), making **A** go **up** (`+`).
- **Reaction 2** points *out of* **A** (conversion of **A** into **B** at rate $k_{1,\text{AtoB}} \cdot A$), making **A** go **down** (`-`).

So the ordinary differential equation for **A** is:

$$
\frac{dA}{dt} = k_{1,\text{Aexp}} - k_{1,\text{AtoB}} \cdot A
$$

With $k_{1,\text{Aexp}} = 1$ and $k_{1,\text{AtoB}} = 2$, substituting those numbers gives:

$$
\frac{dA}{dt} = +1 - 2A
$$

And in Julia code, where `dy[1]` means the rate of change of the 1st species (**A**), `dy[2]` is **B**, and `dy[3]` is **C**, that line looks like:

```julia
dy[1] = + 1 - A * 2
```

## Inspect the generated `toyModel.jl`

Let's run `CSV2Model` and print the exact Julia file it automatically writes for us!

In [1]:
using CSV, DataFrames

# Ensure toyModel.jl is freshly generated in this workspace
run(`python3 csv2model-multiscale.py reactions.csv parameters.csv ratelaws.csv toyModel.jl`)

# Print the entire generated Julia ODE file
print(read("toyModel.jl", String))

inline
Running CSV2JuliaDiffEq with parameters hard-coded into the CSV file, if this is not correct, re-run with the 5th argument set to 'scan'
Opening ratelaws.csv as rate law file
Opening parameters.csv as parameters file
Opening reactions.csv as reactions file
#######################################################
# Generated programmatically by CSV2JuliaDiffEq.      #
# http://github.com/SiFTW/CSV2JuliaDiffEq             #
#######################################################
# generated from:
#    reactions file: reactions.csv
#    parameters file file: parameters.csv
#    rate law file: ratelaws.csv
#
# Statistics:
#    Equations:3
#    Parameters:4
#######################################################



function toyModel(dy,y,p,t)
	A=maximum([y[1],0])
	B=maximum([y[2],0])
	C=maximum([y[3],0])
	#A
	dy[1]= + 1 - A * 2
	#B
	dy[2]= + A * 2 - B * 3
	#C
	dy[3]= + B * 3 - C * 4
end


## Why use three CSV files instead of hand-writing ODEs?

You *could* write `toyModel.jl` by hand directly from a diagram — and many people do for tiny models. But as soon as a biological pathway has 20 or 200 reactions, hand-writing ODEs makes it painfully easy to introduce bugs (for example, subtracting `A * 2` from **A** but forgetting to add `+ A * 2` to **B**, violating conservation of mass!).

Instead, in the Mitchell Lab we define every model in **three clean CSV tables** (Comma-Separated Values — tables you can also open and edit in Excel) and let **`CSV2Model`** generate the Julia equations automatically.

Let's inspect all three files live!

---

### File 1: `reactions.csv` — the network diagram

Think of `reactions.csv` as the **picture of the model**. Every arrow in the diagram becomes one row in this table.

In [2]:
reactions_df = CSV.read("reactions.csv", DataFrame; stripwhitespace=true)
display(reactions_df)

4×5 DataFrame
 Row │ Substrate  Products  Kinetic Law                Modifiers  Parameters
     │ String1?   String1?  String31                   Missing    String15
─────┼───────────────────────────────────────────────────────────────────────
   1 │ missing    A         Constant Exp                 missing  k1_Aexp
   2 │ A          B         Mass Action with substrate   missing  k1_AtoB
   3 │ B          C         Mass Action with substrate   missing  k1_BtoC
   4 │ C          missing   Mass Action with substrate   missing  k1_Cdeg


Notice how each column maps straight back to the diagram:
- **Substrate** & **Products**: Row 1 has no substrate (`missing`) because **A** is synthesized from nothing. Row 4 has no product because **C** degrades away.
- **Kinetic Law**: Names the mathematical formula governing how fast each reaction proceeds (for example, **Mass Action kinetics** or **Michaelis–Menten kinetics** for enzyme-catalyzed reactions).
- **Modifiers**: Empty here because every species is either consumed (substrate) or produced (product). A *modifier* is a molecule that speeds up or inhibits an arrow without being consumed itself (like an enzyme or transcription factor).
- **Parameters**: Gives a descriptive *name* to the numerical rate constant(s) for each reaction (such as `k1_Aexp` or `k1_AtoB`).

---

### File 2: `parameters.csv` — the numerical values

`parameters.csv` is a lookup table that assigns a numerical value to every parameter name used in `reactions.csv`.

In [3]:
parameters_df = CSV.read("parameters.csv", DataFrame; stripwhitespace=true)
display(parameters_df)

4×2 DataFrame
 Row │ Parameter  Value
     │ String15   Int64
─────┼──────────────────
   1 │ k1_Aexp        1
   2 │ k1_AtoB        2
   3 │ k1_BtoC        3
   4 │ k1_Cdeg        4


---

### File 3: `ratelaws.csv` — the mathematical recipes

Every **Kinetic Law** named in `reactions.csv` must have an exact matching row in `ratelaws.csv` (spelling, capitalization, and spaces must match identically!).

In [4]:
ratelaws_df = CSV.read("ratelaws.csv", DataFrame; stripwhitespace=true)
display(ratelaws_df)

2×2 DataFrame
 Row │ Rate Law Name               Rate Law Definition
     │ String31                    String15
─────┼─────────────────────────────────────────────────
   1 │ Constant Exp                {k1}
   2 │ Mass Action with substrate  [S1] * {k1}


Look at the bracket notation inside **Rate Law Definition**:
- **Curly brackets `{ }`** mean *"substitute a parameter here"*. `{k1}` looks up the parameter whose prefix is `k1` from the `Parameters` column in `reactions.csv`, then grabs its number from `parameters.csv`.
- **Square brackets `[ ]`** mean *"substitute a biomolecule here"*. `[S1]` means the **1st substrate** (`[Mod1]` would mean the 1st modifier).

## Putting all three files together

Let's trace what `CSV2Model` did for **A**:

1. **Reaction 1 (`∅ → A`)**:
   - Kinetic Law: `Constant Exp` → `{k1}`
   - Parameter: `k1_Aexp` → `1`
   - Since **A** is a *product*, `CSV2Model` adds `+ 1` to `dy[1]`.
2. **Reaction 2 (`A → B`)**:
   - Kinetic Law: `Mass Action with substrate` → `[S1] * {k1}`
   - Substrate `[S1]` → `A`, Parameter `{k1}` (`k1_AtoB`) → `2`
   - Since **A** is a *substrate*, `CSV2Model` subtracts `- A * 2` from `dy[1]` (and automatically adds `+ A * 2` to `dy[2]` for **B**!).

Combined together, `CSV2Model` produces:

```julia
#A
dy[1] = + 1 - A * 2
#B
dy[2] = + A * 2 - B * 3
#C
dy[3] = + B * 3 - C * 4
```

Now that you understand how the three CSV files generate the entire ODE system, head to **[3. Editing & Extending Models](./03-editing-models-challenges.ipynb)** to modify parameters and add a brand-new species **D**!